# <span style="color: #a855f7">Pandas data analysis</span>

<span style="color: #a855f7">DataFrame, CSV, фільтрація, сортування, нові колонки, groupby і merge на даних</span>

In [1]:
from html import escape

import pandas as pd
from IPython.display import HTML, display


def colorize(value):
    if isinstance(value, pd.DataFrame):
        html = value.to_html(
            index=False, border=0, na_rep="missing", float_format="{:.2f}".format
        )
        html = html.replace("<table ", '<table align="left" ', 1)
        for tag in ("th", "td"):
            html = html.replace(f"<{tag}>", f'<{tag}><span style="color: #a855f7">')
            html = html.replace(f"</{tag}>", f"</span></{tag}>")
        display(HTML(html))
    else:
        display(HTML(f'<pre style="color: #a855f7">{escape(str(value))}</pre>'))

## <span style="color: #a855f7">DataFrame structure</span>

<span style="color: #a855f7">п’ять записів зі словника, shape, назви колонок і head(3)</span>

In [2]:
profiles = pd.DataFrame({
    "name": ["Obsidia", "Selena", "Alice", "Dyrroth", "Helcurt"],
    "age": [25, 19, 25, 16, 45],
    "city": ["Night Palace", "Shadow Swamp", "Sanguine Lair", "Lumina City", "Abyss"],
})
rows, columns = profiles.shape

colorize(profiles)
colorize(f"rows: {rows}\ncolumns: {columns}")
colorize(f"column names: {profiles.columns.tolist()}")
colorize(profiles.head(3))

name,age,city
Obsidia,25,Night Palace
Selena,19,Shadow Swamp
Alice,25,Sanguine Lair
Dyrroth,16,Lumina City
Helcurt,45,Abyss


name,age,city
Obsidia,25,Night Palace
Selena,19,Shadow Swamp
Alice,25,Sanguine Lair


## <span style="color: #a855f7">CSV inspection</span>

<span style="color: #a855f7">читання movies.csv, перші п’ять рядків, shape, dtypes і кількість пропусків через isna().sum()</span>

In [3]:
movies = pd.read_csv("movies.csv", dtype={"year": "int64", "rating": "Float64"})
movie_rows, movie_columns = movies.shape
movie_types = movies.dtypes.astype(str).rename_axis("column").reset_index(name="dtype")
missing_values = movies.isna().sum().rename_axis("column").reset_index(name="missing")

colorize(movies.head(5))
colorize(f"rows: {movie_rows}\ncolumns: {movie_columns}")
colorize(movie_types)
colorize(missing_values)

title,genre,year,rating
Se7en,Crime / Mystery,1995,8.60
The Thing,Horror / Sci-Fi,1982,8.20
Shutter Island,Mystery / Thriller,2010,8.20
The Others,Horror / Mystery,2001,7.60
Zodiac,Crime / Mystery,2007,7.70


column,dtype
title,str
genre,str
year,int64
rating,Float64


column,missing
title,0
genre,0
year,0
rating,1


## <span style="color: #a855f7">Filtering and sorting</span>

<span style="color: #a855f7">булева фільтрація gold > 50000, сортування age за зростанням і gold за спаданням</span>

In [4]:
heroes = profiles[["name", "age"]].copy()
heroes["role"] = ["Marksman", "Assassin / Mage", "Tank / Mage", "Fighter", "Assassin"]
heroes["gold"] = [62000, 47000, 58000, 50000, 67000]

high_gold = heroes.loc[heroes["gold"] > 50000]
by_age = heroes.sort_values("age")
by_gold = heroes.sort_values("gold", ascending=False)

colorize("gold > 50000")
colorize(high_gold)
colorize("age ascending")
colorize(by_age)
colorize("gold descending")
colorize(by_gold)

name,age,role,gold
Obsidia,25,Marksman,62000
Alice,25,Tank / Mage,58000
Helcurt,45,Assassin,67000


name,age,role,gold
Dyrroth,16,Fighter,50000
Selena,19,Assassin / Mage,47000
Obsidia,25,Marksman,62000
Alice,25,Tank / Mage,58000
Helcurt,45,Assassin,67000


name,age,role,gold
Helcurt,45,Assassin,67000
Obsidia,25,Marksman,62000
Alice,25,Tank / Mage,58000
Dyrroth,16,Fighter,50000
Selena,19,Assassin / Mage,47000


## <span style="color: #a855f7">Calculated columns</span>

<span style="color: #a855f7">початкова таблиця з 12 рядками, quantity - кількість елементів у components, revenue = price * quantity, вивід до і після додавання колонки</span>

In [5]:
equipment = pd.DataFrame([
    ("Corrosion Scythe", "Attack", 2050, ["Swift Crossbow", "Regular Spear"]),
    ("Demon Hunter Sword", "Attack", 2180, ["Regular Spear", "Vampire Mallet", "Knife"]),
    ("Golden Staff", "Attack", 2000, ["Regular Spear", "Dagger", "Dagger"]),
    ("Berserker's Fury", "Attack", 2390, ["Legion Sword", "Javelin", "Javelin"]),
    ("Blade of Despair", "Attack", 3010, ["Legion Sword", "Legion Sword"]),
    ("Holy Crystal", "Magic", 3000, ["Magic Wand", "Magic Wand", "Magic Wand"]),
    ("Divine Glaive", "Magic", 1970, ["Magic Wand"]),
    ("Glowing Wand", "Magic", 2050, ["Exotic Veil", "Mystery Codex", "Vitality Crystal"]),
    ("Athena's Shield", "Defense", 2150, ["Silence Robe", "Vitality Crystal"]),
    ("Dominance Ice", "Defense", 2010, ["Black Ice Shield", "Leather Jerkin", "Magic Resist Cloak"]),
    ("Immortality", "Defense", 2120, ["Ares Belt", "Vitality Crystal", "Leather Jerkin"]),
    ("Malefic Roar", "Attack", 2060, ["Legion Sword"]),
], columns=["product", "category", "price", "components"])
items = equipment.drop(columns="components").copy()
items["quantity"] = equipment["components"].map(len)

colorize(items)

product,category,price,quantity
Corrosion Scythe,Attack,2050,2
Demon Hunter Sword,Attack,2180,3
Golden Staff,Attack,2000,3
Berserker's Fury,Attack,2390,3
Blade of Despair,Attack,3010,2
Holy Crystal,Magic,3000,3
Divine Glaive,Magic,1970,1
Glowing Wand,Magic,2050,3
Athena's Shield,Defense,2150,2
Dominance Ice,Defense,2010,3


In [6]:
items["revenue"] = items["price"] * items["quantity"]

colorize(items)

product,category,price,quantity,revenue
Corrosion Scythe,Attack,2050,2,4100
Demon Hunter Sword,Attack,2180,3,6540
Golden Staff,Attack,2000,3,6000
Berserker's Fury,Attack,2390,3,7170
Blade of Despair,Attack,3010,2,6020
Holy Crystal,Magic,3000,3,9000
Divine Glaive,Magic,1970,1,1970
Glowing Wand,Magic,2050,3,6150
Athena's Shield,Defense,2150,2,4300
Dominance Ice,Defense,2010,3,6030


## <span style="color: #a855f7">Grouped statistics</span>

<span style="color: #a855f7">початкова таблиця з 8 рядками, groupby за course, обчислення avg, max і min для score</span>

In [7]:
students = pd.DataFrame([
    ("Obsidia", "Gold lane", 88),
    ("Selena", "Mid lane", 90),
    ("Alice", "EXP lane", 86),
    ("Dyrroth", "EXP lane", 91),
    ("Helcurt", "Jungle", 94),
    ("Selena", "Roam", 95),
    ("Alice", "Mid lane", 87),
    ("Dyrroth", "Jungle", 84),
], columns=["name", "course", "score"])
course_statistics = students.groupby("course", as_index=False).agg(
    avg=("score", "mean"),
    max=("score", "max"),
    min=("score", "min"),
)

colorize(students)
colorize(course_statistics)

name,course,score
Obsidia,Gold lane,88
Selena,Mid lane,90
Alice,EXP lane,86
Dyrroth,EXP lane,91
Helcurt,Jungle,94
Selena,Roam,95
Alice,Mid lane,87
Dyrroth,Jungle,84


course,avg,max,min
EXP lane,88.50,91,86
Gold lane,88.00,88,88
Jungle,89.00,94,84
Mid lane,88.50,90,87
Roam,95.00,95,95


## <span style="color: #a855f7">Joining tables</span>

<span style="color: #a855f7">дві таблиці по п’ять рядків, merge за id з перевіркою one_to_one, total = price * quantity</span>

In [8]:
products = items.loc[:4, ["product", "price"]].rename(columns={"product": "name"}).copy()
products.insert(0, "id", range(1, len(products) + 1))
purchases = pd.DataFrame({
    "id": [3, 1, 5, 2, 4],
    "quantity": [1, 2, 1, 3, 2],
})

colorize(products)
colorize(purchases)

merged = products.merge(purchases, on="id", how="inner", validate="one_to_one")
merged["total"] = merged["price"] * merged["quantity"]

colorize(merged)

id,name,price
1,Corrosion Scythe,2050
2,Demon Hunter Sword,2180
3,Golden Staff,2000
4,Berserker's Fury,2390
5,Blade of Despair,3010


id,quantity
3,1
1,2
5,1
2,3
4,2


id,name,price,quantity,total
1,Corrosion Scythe,2050,2,4100
2,Demon Hunter Sword,2180,3,6540
3,Golden Staff,2000,1,2000
4,Berserker's Fury,2390,2,4780
5,Blade of Despair,3010,1,3010
